In [1]:
import sympy as sp
import numpy as np
from sympy.physics.quantum.cg import CG


def get_basis_info(i):
    if i < 2: return 0, i, 0
    elif i < 11: return 1, (i - 2) // 3, (i - 2) % 3
    else: return 2, (i - 11) // 5, (i - 11) % 5

def build_T_op_exact():
    U_schur = np.zeros((16, 16), dtype=object)
    paths_by_J = {
        0: [(0, sp.Rational(1,2)), (1, sp.Rational(1,2))],
        1: [(0, sp.Rational(1,2)), (1, sp.Rational(1,2)), (1, sp.Rational(3,2))],
        2: [(1, sp.Rational(3,2))]
    }
    col = 0
    for J in [0, 1, 2]:
        M_vals = [sp.Rational(M_2, 2) for M_2 in range(-int(2*J), int(2*J)+2, 2)]
        for s12, s123 in paths_by_J[J]:  
            for M in M_vals:             
                for i in range(16):
                    b = format(i, '04b')
                    m = [sp.Rational(1,2) if char == '0' else sp.Rational(-1,2) for char in b]
                    m12, m123 = m[0]+m[1], m[0]+m[1]+m[2]
                    if m123 + m[3] == M:
                        val = (CG(sp.Rational(1,2), m[0], sp.Rational(1,2), m[1], s12, m12).doit() * CG(s12, m12, sp.Rational(1,2), m[2], s123, m123).doit() * CG(s123, m123, sp.Rational(1,2), m[3], J, M).doit())
                        if val != 0: U_schur[i, col] = val
                col += 1
    U_full = np.kron(U_schur, U_schur)
    sy = np.array([[0, -sp.I], [sp.I, 0]], dtype=object)
    I2 = np.array([[1, 0], [0, 1]], dtype=object)
    S_list = [I2, sy, sy, sy, sy, sy, sy, I2]
    S_full = S_list[0]
    for i in range(1, 8): S_full = np.kron(S_full, S_list[i])
    P_vec = np.zeros((256, 256), dtype=object)
    pv = [0, 1, 3, 5, 2, 4, 6, 7]
    for r_in in range(256):
        b_in = format(r_in, '08b')
        b_out = ''.join([b_in[pv[k]] for k in range(8)])
        P_vec[int(b_out, 2), r_in] = 1
    sort_indices = list(range(256))
    sort_indices.sort(key=lambda x: (get_basis_info(x // 16)[0], get_basis_info(x % 16)[0], 
                                     get_basis_info(x // 16)[2], get_basis_info(x % 16)[2], 
                                     get_basis_info(x // 16)[1], get_basis_info(x % 16)[1]))
    P_sort = np.zeros((256, 256), dtype=object)
    for i_sorted, i_orig in enumerate(sort_indices): P_sort[i_sorted, i_orig] = 1
    T_op = S_full.dot(P_vec.T).dot(U_full).dot(P_sort.T)
    return T_op, np.conjugate(T_op).T

    
def print_non_zero_latex(matrix, items_per_line=7):
    """
    Extract non-zero elements from a matrix and print them in LaTeX align* format.
    """
    print(" -> Extracting non-zero elements and generating LaTeX code...")
    non_zeros = []
    rows, cols = matrix.shape
    
    for i in range(rows):
        for j in range(cols):
            val = matrix[i, j]
            if val != 0:
                val_latex = sp.latex(sp.sympify(val))
                # Convert 0-based to 1-based indexing for math notation
                item_str = f"\\{{{i+1},{j+1}\\}}\\rightarrow{val_latex}"
                non_zeros.append(item_str)
    
    print("\\begin{align*}")
    
    # Print in chunks to avoid overly wide lines
    for idx in range(0, len(non_zeros), items_per_line):
        chunk = non_zeros[idx : idx + items_per_line]
        line = "&" + ", ".join(chunk)
        
        if idx + items_per_line < len(non_zeros):
            line += ", \\\\"
        
        print(line)
        
    print("\\end{align*}")


if __name__ == "__main__":
    # Generate the matrix (may take time due to CG coefficient calculations)
    T_op, T_op_dagger = build_T_op_exact()
    
    print("\n--- Generated LaTeX Code ---")
    print_non_zero_latex(T_op, items_per_line=7)


--- Generated LaTeX Code ---
 -> Extracting non-zero elements and generating LaTeX code...
\begin{align*}
&\{1,53\}\rightarrow\frac{\sqrt{6}}{4}, \{1,56\}\rightarrow\frac{\sqrt{2}}{4}, \{1,59\}\rightarrow\frac{1}{4}, \{1,135\}\rightarrow- \frac{\sqrt{2}}{4}, \{1,136\}\rightarrow- \frac{\sqrt{6}}{12}, \{1,137\}\rightarrow- \frac{\sqrt{3}}{12}, \{1,198\}\rightarrow\frac{\sqrt{3}}{4}, \\
&\{1,238\}\rightarrow- \frac{1}{4}, \{2,132\}\rightarrow- \frac{\sqrt{2}}{2}, \{2,133\}\rightarrow- \frac{\sqrt{6}}{6}, \{2,134\}\rightarrow- \frac{\sqrt{3}}{6}, \{2,237\}\rightarrow- \frac{1}{2}, \{3,34\}\rightarrow\frac{\sqrt{6}}{6}, \{3,36\}\rightarrow\frac{\sqrt{2}}{6}, \\
&\{3,38\}\rightarrow\frac{1}{6}, \{3,61\}\rightarrow- \frac{\sqrt{6}}{6}, \{3,62\}\rightarrow- \frac{\sqrt{3}}{6}, \{3,64\}\rightarrow- \frac{\sqrt{2}}{6}, \{3,65\}\rightarrow- \frac{1}{6}, \{3,67\}\rightarrow- \frac{1}{6}, \{3,68\}\rightarrow- \frac{\sqrt{2}}{12}, \\
&\{3,138\}\rightarrow\frac{\sqrt{3}}{6}, \{3,139\}\rightarrow\fr